# Clasificación estelar D2: código compartido con Airflow

Ejecutar después de **process_etl_star_data**. Esta notebook y el DAG
**train_star_model** importan `airflow/dags/star_training`: preparación D2,
Optuna, entrenamiento, evaluación, registro y comparación/promoción.
No es necesario ejecutar ambos: son dos maneras de ejecutar el mismo flujo.

Nuevas ingestas: SDSS DR17 TOP 1000 con ORDER BY NEWID(), una muestra aleatoria
por consulta, sin balanceo de clases. Para datos nuevos ejecutar primero la ingesta;
repetir solo el entrenamiento reutiliza el CSV. Los snapshots anteriores se conservan.
Predictores: u,g,r,i,z,u-g,g-r,r-i,i-z. Train/test 80/20 estratificado;
CV5 y scaler dentro de cada fold. El test no selecciona hiperparámetros.

La promoción es una celda independiente: solo mejora estricta del F1 weighted
frente al champion sobre el mismo test mueve el alias. Empates y resultados
inferiores conservan la versión anterior. Sin champion se publica el primero válido.

## Dependencias y conexión
Usar el kernel `.venv-star` con `notebooks/requirements_star.txt`.
Iniciar desde la raíz del repo o `notebooks/`. MLflow3.16.1 y sklearn1.7.2.
Windows usa localhost:5001/9000; Docker usa mlflow:5000/s3:9000.
La notebook no carga `.env` automáticamente. Para una prueba corta definir
`STAR_N_TRIALS=2` antes de la configuración. No ejecutar notebook y DAG como
promotores del mismo alias simultáneamente.

In [ ]:
# Si faltan dependencias, instalar requirements_star.txt y reiniciar el kernel.
# %pip install -r requirements_star.txt

In [ ]:
import os
import sys
import hashlib
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'airflow/dags/star_training').is_dir()), None)
if repo is None:
    raise RuntimeError('Abrir Jupyter desde la raíz del repo o notebooks/')
sys.path.insert(0, str(repo / 'airflow/dags'))
from star_training.features import BANDS, D2_FEATURES, CLASSES
from star_training.workflow import (configure_tracking, prepare_data, search_hyperparameters,
    train_and_log_model, evaluate_candidate, register_candidate, compare_and_promote, MODEL_NAME)

SEED = 42
N_TRIALS = int(os.getenv('STAR_N_TRIALS', '250'))
MIN_DELTA = float(os.getenv('STAR_MIN_DELTA', '0'))
MLFLOW_URI = os.getenv('STAR_MLFLOW_URI', 'http://127.0.0.1:5001')
S3_ENDPOINT = os.getenv('STAR_S3_ENDPOINT', 'http://127.0.0.1:9000')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'minio')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'minio123')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
client = configure_tracking(MLFLOW_URI, S3_ENDPOINT)
experiment = mlflow.set_experiment('star type')
experiment_id = experiment.experiment_id
s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT)
print('Experimento:', experiment_id, '| trials:', N_TRIALS)

## Snapshot y preparación compartida
El CSV raw no se normaliza ni sobrescribe. El snapshot se identifica por SHA-256;
objid se conserva como texto. `prepare_data` limpia, calcula colores y separa
train/test con la misma implementación utilizada por Airflow.

In [ ]:
BUCKET = 'data'
raw_bytes = s3.get_object(Bucket=BUCKET, Key='raw/star/starset.csv')['Body'].read()
dataset_sha256 = hashlib.sha256(raw_bytes).hexdigest()
snapshot_key = f'raw/star/snapshots/{dataset_sha256}.csv'
s3.put_object(Bucket=BUCKET, Key=snapshot_key, Body=raw_bytes, ContentType='text/csv')
ingestions = mlflow.search_runs(experiment_ids=[experiment_id],
    filter_string=f"tags.stage = 'ingestion' AND tags.`dataset.sha256` = '{dataset_sha256}'",
    max_results=1)
snapshot = {'bucket':BUCKET, 'key':snapshot_key, 'sha256':dataset_sha256,
            'ingestion_run_id':None if ingestions.empty else ingestions.iloc[0]['run_id']}
data = prepare_data(raw_bytes, seed=SEED)
X_train, X_test, y_train, y_test = data.X_train, data.X_test, data.y_train, data.y_test
print('SHA256:', data.sha256, '| train:', X_train.shape, '| test:', X_test.shape)
display(data.frame['class'].value_counts())
display(data.frame[BANDS + ['class']].head())

In [ ]:
fig_corr, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(X_train.corr(), vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(D2_FEATURES)), D2_FEATURES, rotation=45)
ax.set_yticks(range(len(D2_FEATURES)), D2_FEATURES)
ax.set_title('Correlación D2 — solo entrenamiento')
fig_corr.colorbar(image, ax=ax)
fig_corr.tight_layout()
plt.show()

## Buscar hiperparámetros
250 trials por defecto, F1 weighted CV5 y un run hijo por trial.
La búsqueda se separa de las etapas posteriores; si falla una descarga o
publicación, no repetir esta celda mientras `search` siga en memoria.

In [ ]:
def champion_callback(study, trial):
    if study.best_trial.number == trial.number:
        print(f'Trial {trial.number}: nuevo mejor F1 CV = {trial.value:.4f}')

search = search_hyperparameters(data, snapshot, n_trials=N_TRIALS, callback=champion_callback)
parent_run_id = search['run_id']
print('Run:', parent_run_id)
print('Mejores parámetros:', search['best_params'])
print('F1 CV:', search['best_value'])

## Entrenar y guardar
Reutiliza la selección y el run padre. Si el modelo ya se guardó, recupera el
artefacto READY existente. La preparación y el scaler no se duplican.

In [ ]:
trained = train_and_log_model(data, search)
model_uri = trained['model_uri']
print('Modelo guardado:', model_uri)

## Evaluar el candidato
Recarga el Pipeline guardado, valida el esquema D2 y registra métricas sobre test,
reporte por clase, matriz de confusión y correlación. El scaler sigue dentro del Pipeline.

In [ ]:
evaluated = evaluate_candidate(data, trained)
test_metrics = evaluated['metrics']
loaded = mlflow.sklearn.load_model(model_uri)
display(pd.Series(test_metrics))
display(pd.DataFrame({'real':y_test.head(5), 'prediccion':loaded.predict(X_test.head(5))}))

## Registrar candidato
Registra una versión sin mover el alias. Reejecutar con el mismo run/artefacto
recupera la versión existente, evitando duplicarla.

In [ ]:
candidate = register_candidate(evaluated)
print('Candidato:', candidate['name'], 'versión', candidate['version'])

## Comparar y promover automáticamente
Equivale a la tarea independiente `compare_and_promote_champion` de Airflow.
Evalúa al candidato y a la versión champion actual sobre exactamente el mismo test.
Solo se promueve si su F1 weighted es estrictamente mayor al del champion más
`MIN_DELTA` (0 por defecto). Ante empate o peor resultado, conserva champion.
Si no existe champion, promueve el primero válido. Un error de carga o esquema
interrumpe la comparación antes de mover el alias.

El resultado queda en `promotion/comparison.json` del run en MLflow.
No es una validación científica independiente: el test se usa para decidir
promociones, y un champion de otro dataset puede haber visto objetos coincidentes.

In [ ]:
promotion = compare_and_promote(data, candidate, min_delta=MIN_DELTA)
display(pd.Series(promotion))

## Predecir con el champion vigente
Si el candidato no mejoró, esta celda carga la versión anterior. La web revisa
el alias cada minuto. Las nueve entradas se entregan sin normalizar.

In [ ]:
champion = mlflow.sklearn.load_model(f'models:/{MODEL_NAME}@champion')
new_bands = pd.DataFrame([{'u':20.1, 'g':19.3, 'r':18.8, 'i':18.5, 'z':18.3}])
new_d2 = new_bands.copy()
for left, right in zip(BANDS[:-1], BANDS[1:]):
    new_d2[f'{left}-{right}'] = new_d2[left] - new_d2[right]
print(champion.predict(new_d2[D2_FEATURES]))